# Expert ML Pipeline: Regression — `Irrigation_Requirement_mm`

> **Objective:** Predict irrigation requirements

| | |
|---|---|
| **Dataset** | `train.csv` |
| **Target** | `Irrigation_Requirement_mm` |
| **Problem type** | Regression |
| **Rows / Columns** | 10,000 / 18 |
| **Data quality** | 87/100 (Grade B) |

---
*Auto-generated by the Agentic ML Pipeline. All preprocessing decisions, model selection,
and hyperparameter tuning are driven by data-analysis agents. Run cells top-to-bottom.*

## Setup

Import all libraries needed for the full ML pipeline.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, KFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    r2_score, mean_absolute_error, mean_squared_error,
)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import (
    LogisticRegression, Ridge, Lasso, ElasticNet, RidgeClassifier,
    LinearRegression,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier,    RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    ExtraTreesClassifier,      ExtraTreesRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
    AdaBoostClassifier,        AdaBoostRegressor,
    BaggingClassifier,         BaggingRegressor,
)
from sklearn.svm import LinearSVC, SVC, LinearSVR, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("optuna not installed — run: pip install optuna")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RANDOM_STATE = 42
print("Environment ready.")

## 1. Data Loading & Initial Exploration

In [ ]:
TARGET       = 'Irrigation_Requirement_mm'
TRAIN_PATH   = r'train.csv'
TEST_PATH    = r'test.csv'   # update if your test file has a different path
PROBLEM_TYPE = 'regression'

df = pd.read_csv(TRAIN_PATH)
print(f"Training set shape: {df.shape}")
print(f"Target column  : {TARGET}")
print(f"Problem type   : {PROBLEM_TYPE}")
df.head()

In [ ]:
# Overview: dtypes, missing values, basic stats
print("=== Column dtypes ===")
print(df.dtypes.value_counts())

print("\n=== Missing values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct (%)": missing_pct})
missing_df = missing_df[missing_df["count"] > 0].sort_values("count", ascending=False)

if not missing_df.empty:
    display(missing_df)
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.35)))
    missing_df["pct (%)"].plot(kind="barh", ax=ax, color="#ef4444")
    ax.set_xlabel("Missing %")
    ax.set_title("Missing Values per Column", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("  No missing values found!")

print("\n=== Basic statistics ===")
display(df.describe(include="all").T)

## 2. Exploratory Data Analysis

### Key insights from agent analysis

- Temperature_C strongly correlated with target (r=0.72)

In [ ]:
# ── Target distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[TARGET].dropna(), bins=50, color="#6366f1", edgecolor="white", alpha=0.8)
axes[0].set_title(f"Distribution: {TARGET}", fontweight="bold")
axes[0].set_xlabel(TARGET); axes[0].set_ylabel("Count")

axes[1].boxplot(df[TARGET].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor="#6366f1", alpha=0.7))
axes[1].set_title(f"Spread: {TARGET}", fontweight="bold")
axes[1].set_ylabel(TARGET)

plt.suptitle("Target Variable Analysis", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()
print(df[TARGET].describe())

In [ ]:
# ── Top significant feature distributions ────────────────────────
TOP_FEATURES = ['Temperature_C']
TOP_FEATURES = [f for f in TOP_FEATURES if f in df.columns][:6]

if TOP_FEATURES:
    n_cols = 3
    n_rows = (len(TOP_FEATURES) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
    axes_flat = np.array(axes).flatten()

    for i, col in enumerate(TOP_FEATURES):
        ax = axes_flat[i]
        if pd.api.types.is_numeric_dtype(df[col]):
            ax.hist(df[col].dropna(), bins=30, density=True,
                    alpha=0.7, color="#6366f1", edgecolor="white")
            ax.set_title(col, fontweight="bold", fontsize=11)
            ax.set_xlabel(col); ax.set_ylabel("Density")
        else:
            vc = df[col].value_counts().head(10)
            ax.bar(vc.index.astype(str), vc.values, color="#6366f1", alpha=0.8)
            ax.set_title(col, fontweight="bold", fontsize=11)
            ax.set_xlabel(col); ax.set_ylabel("Count")
            ax.tick_params(axis="x", rotation=45)

    for j in range(len(TOP_FEATURES), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle("Top Feature Distributions", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(num_cols) >= 2:
    target_in_num = TARGET in num_cols
    if target_in_num:
        tc_abs = df[num_cols].corr()[TARGET].abs().sort_values(ascending=False)
        plot_cols = tc_abs.head(20).index.tolist()
    else:
        plot_cols = num_cols[:20]

    corr = df[plot_cols].corr()
    fig, axes = plt.subplots(1, 2 if target_in_num else 1,
                              figsize=(18 if target_in_num else 10, 7))
    if not target_in_num:
        axes = [axes]

    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
                annot=len(plot_cols) <= 12, fmt=".2f",
                ax=axes[0], square=True, linewidths=0.5)
    axes[0].set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")

    if target_in_num:
        tc = df[num_cols].corr()[TARGET].drop(TARGET).abs() \
               .sort_values(ascending=False).head(15)
        colors = ["#22c55e" if v > 0.3 else "#f59e0b" if v > 0.1 else "#64748b"
                  for v in tc.values]
        axes[1].barh(tc.index, tc.values, color=colors)
        axes[1].set_title(f"Correlation with {TARGET}", fontsize=13, fontweight="bold")
        axes[1].set_xlabel("Absolute Correlation")
        axes[1].axvline(0.3, color="green", ls="--", alpha=0.7, label="Strong (0.3)")
        axes[1].axvline(0.1, color="orange", ls="--", alpha=0.7, label="Moderate (0.1)")
        axes[1].legend()

    plt.tight_layout(); plt.show()

## 3. Statistical Feature Significance

Statistical tests (t-test / Mann-Whitney / ANOVA / Kruskal-Wallis / Chi-square / Correlation)
were run on every feature. Benjamini-Hochberg FDR correction was applied.

**Significant (2 features, p < 0.05 after FDR):**

  - `Temperature_C`
  - `Soil_Moisture`

**Large effect size:** `Temperature_C`
**Medium effect size:** `Soil_Moisture`
**Insignificant / deprioritised:** 1 features

In [ ]:
# ── Feature significance visualisation ───────────────────────────
ranked_features = [
  {
    "feature": "Temperature_C",
    "effect_size": 0.72,
    "p_value": 0.0001
  }
]
large_effect    = ["Temperature_C"]
medium_effect   = ["Soil_Moisture"]

if ranked_features:
    rdf = pd.DataFrame(ranked_features)

    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(rdf) * 0.4)))

    if "effect_size" in rdf.columns:
        rdf_s = rdf.sort_values("effect_size", ascending=True).tail(15)
        colors = [
            "#22c55e" if r in large_effect else
            "#f59e0b" if r in medium_effect else "#64748b"
            for r in rdf_s["feature"].tolist()
        ]
        axes[0].barh(rdf_s["feature"].astype(str), rdf_s["effect_size"], color=colors)
        axes[0].set_title("Feature Effect Size", fontsize=13, fontweight="bold")
        axes[0].set_xlabel("Effect Size")
        from matplotlib.patches import Patch
        axes[0].legend(handles=[
            Patch(color="#22c55e", label="Large"),
            Patch(color="#f59e0b", label="Medium"),
            Patch(color="#64748b", label="Small"),
        ])

    if "p_value" in rdf.columns:
        rdf["neg_log_p"] = -np.log10(rdf["p_value"].clip(lower=1e-300))
        rdf_s2 = rdf.sort_values("neg_log_p", ascending=True).tail(15)
        c2 = ["#22c55e" if p < 0.01 else "#f59e0b" if p < 0.05 else "#64748b"
               for p in rdf_s2["p_value"]]
        axes[1].barh(rdf_s2["feature"].astype(str), rdf_s2["neg_log_p"], color=c2)
        axes[1].axvline(-np.log10(0.05), color="orange", ls="--", label="p = 0.05")
        axes[1].set_title("-log₁₀(p-value)", fontsize=13, fontweight="bold")
        axes[1].set_xlabel("-log₁₀(p)")
        axes[1].legend()

    plt.suptitle("Statistical Feature Significance", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("No ranked-feature data available.")

## 4. Data Preprocessing

### Agent preprocessing decisions

The preprocessing agent analysed the data and applied:

  - Imputed Missing Values
  - Scaled Numeric Features

| Metric | Value |
|--------|-------|
| Original features | 17 |
| Dropped (bad/irrelevant) | 1 |
| Log-transformed | 1 |
| Frequency encoded | 0 |
| One-hot encoded | 1 |
| **Final feature count** | **15** |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Preprocessing Configuration  (derived from agent analysis)
# ═══════════════════════════════════════════════════════════════

COLS_TO_DROP      = ['id']
LOG_TRANSFORM_COLS= ['Rainfall_mm']
WINSORIZE_COLS    = ['Rainfall_mm']   # capped before log-transform
FREQ_ENCODE_COLS  = []
OHE_COLS          = ['Soil_Type']

print(f"Drop        : {len(COLS_TO_DROP)} cols")
print(f"Log-transform : {len(LOG_TRANSFORM_COLS)} cols")
print(f"Freq-encode  : {len(FREQ_ENCODE_COLS)} cols")
print(f"One-hot      : {len(OHE_COLS)} cols")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# fit_preprocessor / apply_preprocessor
# ═══════════════════════════════════════════════════════════════

def fit_preprocessor(df, target_col=TARGET):
    """Fit on training data. Returns (X, y, transformers)."""
    df = df.copy()

    # 1. Drop irrelevant columns
    drop = [c for c in COLS_TO_DROP if c in df.columns and c != target_col]
    if drop:
        df = df.drop(columns=drop)

    y = df[target_col].copy() if target_col in df.columns else None
    X = df.drop(columns=[target_col], errors="ignore")

    # 2. Impute missing values
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    medians = {c: X[c].median() for c in num_cols if X[c].isnull().any()}
    modes   = {c: X[c].mode()[0] for c in cat_cols if X[c].isnull().any()}
    for c, v in medians.items(): X[c] = X[c].fillna(v)
    for c, v in modes.items():   X[c] = X[c].fillna(v)

    # 3. Winsorise (IQR-based outlier capping)
    iqr_bounds = {}
    for c in WINSORIZE_COLS:
        if c in X.columns and pd.api.types.is_numeric_dtype(X[c]):
            q1, q3 = X[c].quantile(0.25), X[c].quantile(0.75)
            iqr = q3 - q1
            if iqr > 0:
                lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                iqr_bounds[c] = (lo, hi)
                X[c] = X[c].clip(lower=lo, upper=hi)

    # 4. Log-transform skewed columns
    for c in LOG_TRANSFORM_COLS:
        if c in X.columns:
            X[c] = np.log1p(X[c].clip(lower=0))

    # 5. Frequency encode high-cardinality categoricals
    freq_maps = {}
    for c in FREQ_ENCODE_COLS:
        if c in X.columns:
            fm = X[c].value_counts(normalize=True).to_dict()
            freq_maps[c] = fm
            X[f"{c}_freq"] = X[c].map(fm).fillna(0.0)
            X = X.drop(columns=[c])

    # 6. One-hot encode remaining categoricals
    ohe_present = [c for c in OHE_COLS if c in X.columns]
    if ohe_present:
        X = pd.get_dummies(X, columns=ohe_present, drop_first=True)

    feature_names = X.columns.tolist()

    # 7. Scale all numeric features
    scale_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    X[scale_cols] = scaler.fit_transform(X[scale_cols])

    transformers = dict(
        medians=medians, modes=modes,
        iqr_bounds=iqr_bounds, freq_maps=freq_maps,
        feature_names=feature_names,
        scaler=scaler, scale_cols=scale_cols,
    )
    return X, y, transformers


def apply_preprocessor(df, transformers, target_col=TARGET):
    """Apply pre-fitted transformers to new data (test set)."""
    df = df.copy()
    drop = [c for c in COLS_TO_DROP if c in df.columns and c != target_col]
    if drop:
        df = df.drop(columns=drop)

    y = df[target_col].copy() if target_col in df.columns else None
    X = df.drop(columns=[target_col], errors="ignore")

    for c, v in transformers["medians"].items():
        if c in X.columns: X[c] = X[c].fillna(v)
    for c, v in transformers["modes"].items():
        if c in X.columns: X[c] = X[c].fillna(v)

    for c, (lo, hi) in transformers["iqr_bounds"].items():
        if c in X.columns: X[c] = X[c].clip(lower=lo, upper=hi)

    for c in LOG_TRANSFORM_COLS:
        if c in X.columns: X[c] = np.log1p(X[c].clip(lower=0))

    for c, fm in transformers["freq_maps"].items():
        if c in X.columns:
            X[f"{c}_freq"] = X[c].map(fm).fillna(0.0)
            X = X.drop(columns=[c])

    ohe_present = [c for c in OHE_COLS if c in X.columns]
    if ohe_present:
        X = pd.get_dummies(X, columns=ohe_present, drop_first=True)

    X = X.reindex(columns=transformers["feature_names"], fill_value=0)

    sc_cols = [c for c in transformers["scale_cols"] if c in X.columns]
    if sc_cols:
        X[sc_cols] = transformers["scaler"].transform(X[sc_cols])

    return X, y


# Apply to full training data
X_full, y_full, transformers = fit_preprocessor(df)
print(f"Preprocessed training shape : {X_full.shape}")
print(f"Feature names (first 8)     : {X_full.columns.tolist()[:8]}")
X_full.head()

## 5. Baseline Model Benchmarking

Quick evaluation of simple models to bound performance before tuning.

**Agent baseline result:**
- Best model: `Ridge`
- `r2` = `0.6100`
- Reasoning: Best linear baseline.

In [ ]:
# ── Train / validation split ──────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.20, random_state=RANDOM_STATE, stratify=None
)
print(f"Train : {X_train.shape}   Val : {X_val.shape}")

# Agent's candidate results for reference
AGENT_BASELINE = []

In [ ]:
# ── Evaluate candidates ───────────────────────────────────────────
def score_model(m, Xtr, Xv, ytr, yv):
    m.fit(Xtr, ytr); p = m.predict(Xv)
    return {"r2": round(r2_score(yv, p), 4),
            "mae": round(mean_absolute_error(yv, p), 4)}
SCORE_KEY = "r2"

candidates = [
    ("Dummy (mean)",      DummyRegressor(strategy="mean")),
    ("Linear Regression", LinearRegression()),
    ("Ridge",             Ridge(random_state=RANDOM_STATE)),
    ("Lasso",             Lasso(random_state=RANDOM_STATE, max_iter=5000)),
    ("Decision Tree",     DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)),
    ("Random Forest",     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ("Hist GBM",          HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
]

print("\nTraining & evaluating baseline models …")
results = []
for name, model in candidates:
    try:
        s = score_model(model, X_train, X_val, y_train, y_val)
        results.append({"model": name, **s})
        print(f"  {name:<32} {SCORE_KEY}={s[SCORE_KEY]:.4f}")
    except Exception as exc:
        print(f"  {name:<32} FAILED: {exc}")

res_df = pd.DataFrame(results).sort_values(SCORE_KEY, ascending=False)
print("\n=== Baseline Results ===")
display(res_df)

# Visualise
fig, ax = plt.subplots(figsize=(10, max(4, len(res_df) * 0.5)))
colors = ["#22c55e" if i == 0 else "#6366f1" for i in range(len(res_df))]
ax.barh(res_df["model"], res_df[SCORE_KEY], color=colors)
ax.set_xlabel(SCORE_KEY.upper())
ax.set_title(f"Baseline Model Comparison ({SCORE_KEY.upper()})", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

## 6. Hyperparameter Tuning

Bayesian optimisation with Optuna (TPE sampler) over the most promising model families.

**Agent best result:**
- Model: `HistGradientBoostingRegressor`
- CV `r2` = `0.8300`
- Best params: `{"learning_rate": 0.08, "max_depth": 6}`

In [ ]:
# ── Agent tuning results (reference) ─────────────────────────────
AGENT_TUNED   = []
AGENT_BEST_PARAMS = {
    "learning_rate": 0.08,
    "max_depth": 6
}

print("Agent best model  :", 'HistGradientBoostingRegressor')
print("Agent best params :", AGENT_BEST_PARAMS)
print(f"Agent CV r2    : 0.8300")

# ── Run Optuna study (or skip and use AGENT_BEST_PARAMS) ──────────
N_TRIALS  = 30
CV_FOLDS  = 5
cv_split  = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def optuna_objective(trial):
    params = {
        "learning_rate":  trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":      trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12]),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 127, step=8),
        "min_samples_leaf":trial.suggest_int("min_samples_leaf", 5, 50, step=5),
    }
    model = HistGradientBoostingRegressor(**params, random_state=RANDOM_STATE)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv_split, scoring='r2', n_jobs=-1,
    )
    return float(scores.mean())

if HAS_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nOptuna best r2: {study.best_value:.4f}")
    print(f"Optuna best params : {study.best_params}")
    BEST_PARAMS = {**AGENT_BEST_PARAMS, **study.best_params}
else:
    print("Optuna not available — using agent's pre-tuned parameters.")
    BEST_PARAMS = AGENT_BEST_PARAMS

print(f"\nFinal hyperparameters: {BEST_PARAMS}")

## 7. Best Model — Full Evaluation

| | |
|---|---|
| **Model** | `HistGradientBoostingRegressor` |
| **Primary metric (r2)** | `0.8400` |
| **sklearn class** | `HistGradientBoostingRegressor` |

Training on the 80 % train split and evaluating on the 20 % validation split.

In [ ]:
# ── Instantiate best model ────────────────────────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor

best_params_final = BEST_PARAMS if "BEST_PARAMS" in dir() else {
    "learning_rate": 0.08,
    "max_depth": 6
}

# Strip pipeline-prefix keys and known-fixed params
_SKIP = {"random_state", "n_jobs", "max_iter", "early_stopping", "eval_metric"}
clean_p = {k.replace("model__", ""): v
           for k, v in best_params_final.items()
           if k.replace("model__", "") not in _SKIP}

try:
    best_model = HistGradientBoostingRegressor(
        **clean_p,
        random_state=RANDOM_STATE,

    )
except TypeError:
    best_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE)

# Fit and evaluate
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_val)

if PROBLEM_TYPE == "classification":
    val_acc = accuracy_score(y_val, y_pred)
    val_f1  = f1_score(y_val, y_pred, average="weighted", zero_division=0)
    print(f"Validation accuracy    : {val_acc:.4f}")
    print(f"Validation F1 weighted : {val_f1:.4f}")
    print("\n" + classification_report(y_val, y_pred))
else:
    val_r2   = r2_score(y_val, y_pred)
    val_mae  = mean_absolute_error(y_val, y_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"Validation R²   : {val_r2:.4f}")
    print(f"Validation MAE  : {val_mae:.4f}")
    print(f"Validation RMSE : {val_rmse:.4f}")

In [ ]:
# ── Evaluation plots ─────────────────────────────────────────────
if PROBLEM_TYPE == "classification":
    cm = confusion_matrix(y_val, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, cmap="Blues", colorbar=True)
    ax.set_title(f"Confusion Matrix — HistGradientBoostingRegressor", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(y_val, y_pred, alpha=0.4, color="#6366f1", edgecolors="none")
    lo = min(y_val.min(), y_pred.min()); hi = max(y_val.max(), y_pred.max())
    axes[0].plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect fit")
    axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
    axes[0].set_title("Actual vs Predicted", fontweight="bold"); axes[0].legend()

    residuals = y_val - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.4, color="#f59e0b", edgecolors="none")
    axes[1].axhline(0, color="r", ls="--", lw=2)
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")
    axes[1].set_title("Residual Plot", fontweight="bold")
    plt.suptitle(f"Model Evaluation — R² = {val_r2:.4f}", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Feature importance (if available) ────────────────────────────
inner = getattr(best_model, "named_steps", {"model": best_model}).get("model", best_model)
feat_names = X_full.columns.tolist()

if hasattr(inner, "feature_importances_"):
    imp = pd.Series(inner.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, max(5, len(imp) * 0.4)))
    imp.sort_values().plot(kind="barh", ax=ax, color="#6366f1", edgecolor="white")
    ax.set_title("Top Feature Importances", fontsize=13, fontweight="bold")
    ax.set_xlabel("Importance"); plt.tight_layout(); plt.show()

elif hasattr(inner, "coef_"):
    coef = np.abs(inner.coef_).mean(axis=0) if inner.coef_.ndim > 1 else np.abs(inner.coef_)
    n = min(len(feat_names), len(coef))
    cs = pd.Series(coef[:n], index=feat_names[:n]).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, max(5, len(cs) * 0.4)))
    cs.sort_values().plot(kind="barh", ax=ax, color="#6366f1", edgecolor="white")
    ax.set_title("Feature Coefficients (Absolute)", fontsize=13, fontweight="bold")
    ax.set_xlabel("|Coefficient|"); plt.tight_layout(); plt.show()

## 8. Test Data Pipeline & Final Predictions

Retrain on the **full** training set, apply the same preprocessing to test data, and generate predictions.

In [ ]:
# ── Load & preprocess test data ───────────────────────────────────
if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"Test set shape: {test_df.shape}")
    X_test, _ = apply_preprocessor(test_df, transformers, target_col=TARGET)
    print(f"Preprocessed test: {X_test.shape}")
    HAS_TEST = True
else:
    print(f"Test file not found at: {TEST_PATH}")
    print("Using validation split as demonstration.")
    test_df = df.iloc[int(0.8 * len(df)):].reset_index(drop=True)
    X_test, _ = apply_preprocessor(test_df, transformers, target_col=TARGET)
    HAS_TEST = False

# ── Retrain best model on ALL training data ───────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor

_SKIP = {"random_state", "n_jobs", "max_iter", "early_stopping", "eval_metric"}
clean_final = {k.replace("model__", ""): v
               for k, v in (BEST_PARAMS if "BEST_PARAMS" in dir() else {}).items()
               if k.replace("model__", "") not in _SKIP}

try:
    final_estimator = HistGradientBoostingRegressor(
        **clean_final,
        random_state=RANDOM_STATE,

    )
except TypeError:
    final_estimator = HistGradientBoostingRegressor(random_state=RANDOM_STATE)

final_pipeline = final_estimator
final_pipeline.fit(X_full, y_full)

final_preds = final_pipeline.predict(X_test)
print(f"Predictions shape: {final_preds.shape}")
print(f"Sample            : {final_preds[:10]}")

In [ ]:
# ── Build submission CSV ─────────────────────────────────────────
id_col = next(
    (c for c in ["id", "ID", "Id", "row_id", "PassengerId"] if c in test_df.columns),
    None,
)

if id_col:
    submission = pd.DataFrame({id_col: test_df[id_col], TARGET: final_preds})
else:
    submission = pd.DataFrame({"id": range(len(final_preds)), TARGET: final_preds})

submission.to_csv("submission.csv", index=False)
print("Saved  : submission.csv")
print(f"Shape  : {submission.shape}")
display(submission.head(10))

## 9. Results & Next Steps

| | |
|---|---|
| **Best model** | `HistGradientBoostingRegressor` |
| **Primary metric (r2)** | `0.8400` |
| **Baseline score** | `0.6100` |
| **Improvement over baseline** | `+0.2300` |

### Suggestions for further improvement

- **Feature engineering**: polynomial interactions, domain-specific features
- **Ensemble / stacking**: blend top-3 models
- **Cross-validation**: use full CV (5-fold) for final score estimates
- **Calibration** (classification): `CalibratedClassifierCV` for probability outputs
- **Domain knowledge**: add external data sources

In [ ]:
# ── Final summary ─────────────────────────────────────────────────
print("=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)
print(f"  Best model   : HistGradientBoostingRegressor")
print(f"  Metric       : r2")
print(f"  Agent score  : 0.8400")

if PROBLEM_TYPE == "classification":
    print(f"  Val accuracy : {accuracy_score(y_val, y_pred):.4f}")
    print(f"  Val F1-w     : {f1_score(y_val, y_pred, average='weighted', zero_division=0):.4f}")
else:
    print(f"  Val R²       : {r2_score(y_val, y_pred):.4f}")
    print(f"  Val MAE      : {mean_absolute_error(y_val, y_pred):.4f}")
    print(f"  Val RMSE     : {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")

print("=" * 60)
print("Notebook complete!  submission.csv is ready.")